In [15]:
import os
import scanpy as sc
import celltypist
import pandas as pd
import matplotlib.pyplot as plt
import warnings
import tarfile
import urllib.request
import shutil
import glob
import ssl

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# Scanpy settings
sc.settings.verbosity = 3
sc.logging.print_header()
sc.settings.set_figure_params(dpi=100, facecolor='white', frameon=False)

print("Environment setup complete.")

Environment setup complete.


In [16]:
# --- 1. Setup Data Directory ---
os.makedirs("data", exist_ok=True)

# --- 2. Download Whitelist (With SSL Fix) ---
url = "https://github.com/f0t1h/3M-february-2018/raw/refs/heads/master/3M-february-2018.txt.gz"
whitelist_path = "data/whitelist.txt.gz"

print(f"Downloading whitelist from {url}...")

# Create an unverified SSL context to bypass macOS certificate errors
ssl_context = ssl.create_default_context()
ssl_context.check_hostname = False
ssl_context.verify_mode = ssl.CERT_NONE

try:
    with urllib.request.urlopen(url, context=ssl_context) as response, open(whitelist_path, 'wb') as out_file:
        shutil.copyfileobj(response, out_file)
    print("Whitelist download successful.")
except Exception as e:
    print(f"Download failed: {e}")

# Unzip the whitelist
if os.path.exists(whitelist_path):
    !gunzip -f {whitelist_path}
    print("Whitelist extracted.")

# --- 3. Extract/Check Input Data ---
tar_filename = "toy_read_ref_set.tar.gz"
if os.path.exists(tar_filename):
    print(f"Found {tar_filename}, extracting...")
    with tarfile.open(tar_filename, "r:gz") as tar:
        tar.extractall()
    print("Extraction complete.")

# --- 4. Auto-Rename Files to Standard Names ---
def find_and_rename(extension, target_name):
    files = glob.glob(f"**/*{extension}", recursive=True)
    # Filter out files already in data/ or correctly named
    files = [f for f in files if "data/" not in f and f != target_name]
    if files:
        print(f"Found {files[0]} -> Renaming to {target_name}")
        shutil.move(files[0], target_name)
    elif os.path.exists(target_name):
        print(f"{target_name} is present.")
    else:
        print(f"WARNING: Could not find file for {target_name}")

find_and_rename(".fa", "genome.fa")
find_and_rename(".gtf", "genes.gtf")

# Identify Read 1 and Read 2
fastqs = sorted(glob.glob("**/*.fastq.gz", recursive=True))
fastqs = [f for f in fastqs if f != "read1.fastq.gz" and f != "read2.fastq.gz"]

if len(fastqs) >= 2:
    print(f"Found read 1: {fastqs[0]} -> Renaming to read1.fastq.gz")
    shutil.move(fastqs[0], "read1.fastq.gz")
    print(f"Found read 2: {fastqs[1]} -> Renaming to read2.fastq.gz")
    shutil.move(fastqs[1], "read2.fastq.gz")
elif os.path.exists("read1.fastq.gz"):
    print("Reads already renamed.")

Whitelist download successful.
Whitelist extracted.
Found toy_read_ref_set.tar.gz, extracting...
Extraction complete.
Found toy_ref_read/toy_human_ref/fasta/genome.fa -> Renaming to genome.fa
Found toy_ref_read/toy_human_ref/genes/genes.gtf -> Renaming to genes.gtf


In [17]:
%%bash
set -e  # Stop immediately if any command fails

# --- Configuration ---
GENOME="genome.fa"
GTF="genes.gtf"
R1="read1.fastq.gz"
R2="read2.fastq.gz"

# Outputs
IDX="data/salmon_index"
MAP_OUT="data/alevin_out"

echo "--- Checking Inputs ---"
ls -lh $GENOME $GTF $R1 $R2

# --- Step 1: Generate Transcript-to-Gene (t2g) Map ---
echo "Generating t2g.tsv from GTF..."
grep 'transcript_id' $GTF | \
awk -F';' '{print $1, $3}' | \
sed 's/transcript_id "//' | sed 's/"; gene_id "/\t/' | sed 's/"//' > data/t2g.tsv

# Verify t2g is not empty
if [ ! -s data/t2g.tsv ]; then
    echo "ERROR: data/t2g.tsv is empty! Check GTF format."
    exit 1
fi

# --- Step 2: Build Salmon Index ---
echo "Building Salmon index..."
# We use -i for the index folder
salmon index -t $GENOME -i $IDX -p 2

# --- Step 3: Run Salmon Alevin (Mapping) ---
echo "Running Salmon Alevin mapping..."
salmon alevin -l ISR -1 $R1 -2 $R2 \
  -i $IDX \
  --tgMap data/t2g.tsv \
  --output $MAP_OUT \
  --rad --sketch \
  -p 2

echo "--- Mapping Complete ---"
ls -lh $MAP_OUT

Generating t2g map...
Building Salmon index...
zsh:1: command not found: salmon
Running Salmon Alevin mapping...
zsh:1: command not found: salmon


In [ ]:
%%bash
set -e  # Stop immediately on error

MAP_OUT="data/alevin_out"
QUANT_OUT="data/fry_quant"
WHITELIST="data/whitelist.txt"
T2G="data/t2g.tsv"

# --- Step 1: Generate Permit List ---
echo "Generating permit list..."
alevin-fry generate-permit-list -d forward -i $MAP_OUT -o $QUANT_OUT -u $WHITELIST

# --- Step 2: Collate ---
echo "Collating records..."
alevin-fry collate -i $QUANT_OUT -r $MAP_OUT -t 2

# --- Step 3: Quantify ---
echo "Quantifying..."
alevin-fry quant -i $QUANT_OUT -o $QUANT_OUT/res -t 2 -r cr-like -m $T2G --use-mtx

# --- Step 4: Verify and Compress Output ---
RES_DIR="$QUANT_OUT/res/alevin"
echo "Checking output in $RES_DIR..."
ls -F $RES_DIR

# Check if matrix.mtx exists (uncompressed) and zip it if needed
if [ -f "$RES_DIR/matrix.mtx" ]; then
    echo "Compressing matrix.mtx to matrix.mtx.gz..."
    gzip -f "$RES_DIR/matrix.mtx"
fi

# Verify the final file exists for Scanpy
if [ ! -f "$RES_DIR/matrix.mtx.gz" ]; then
    echo "ERROR: matrix.mtx.gz not found in $RES_DIR!"
    exit 1
fi

echo "SUCCESS: Matrix generated and verified."

In [ ]:
# 1. Load Data
print("Loading count matrix...")
# We point to the output folder from the previous step
adata = sc.read_10x_mtx(
    'data/fry_quant/res/alevin',
    var_names='gene_symbols', 
    cache=True
)

# 2. Quality Control (QC)
# Calculate mitochondrial content
adata.var['mt'] = adata.var_names.str.startswith('MT-') 
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

print(f"Cells before filtering: {adata.n_obs}")

# Filter cells/genes
sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=3)
# Filter high MT content
adata = adata[adata.obs.pct_counts_mt < 5, :]

print(f"Cells after filtering: {adata.n_obs}")

# 3. Normalization & Log Transformation
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

# 4. Dimensionality Reduction & Clustering
sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.5)
adata = adata[:, adata.var.highly_variable]
sc.pp.scale(adata, max_value=10)
sc.tl.pca(adata, svd_solver='arpack')
sc.pp.neighbors(adata, n_neighbors=10, n_pcs=40)
sc.tl.umap(adata)
sc.tl.leiden(adata)

# Plot Clustering
sc.pl.umap(adata, color=['leiden'], title="Leiden Clustering", show=True)

In [ ]:
# 1. Load Model
print("Loading CellTypist model...")
# Downloads 'Immune_All_Low.pkl' if not present
model = celltypist.models.Model.load(model='Immune_All_Low.pkl')

# 2. Annotate
print("Annotating cells...")
predictions = celltypist.annotate(adata, model='Immune_All_Low.pkl', majority_voting=True)

# 3. Store results
adata.obs['cell_type'] = predictions.predicted_labels['predicted_labels']
adata.obs['conf_score'] = predictions.predicted_labels['conf_score']

# 4. Plot Annotation
sc.pl.umap(adata, color=['cell_type'], title="CellTypist Annotation", legend_loc='on data')

print("\nDetected Cell Types:")
print(adata.obs['cell_type'].value_counts())